# Azure OpenAI (APIM) Chat Demo

This notebook is a walkthrough version of `Lab2.py`. It shows how to call an **Azure OpenAI**
model through an **Azure API Management (APIM)** gateway using the standard `openai` Python
SDK, and it repeats the same conversational flow four times to demonstrate a simple
"ask → answer → self-evaluate" pattern:

1. Ask the model for a fun fact.
2. Ask the model to invent a hard, IQ-style question.
3. Ask the model to answer its own question.
4. Ask the model to evaluate whether that answer was correct.

Each code cell below is preceded by an explanation of what it does and why.

## 1. Imports

- `dotenv.load_dotenv` reads key/value pairs from a local `.env` file into environment variables.
- `AzureOpenAI` is the Azure-flavored client class from the `openai` package (as opposed to the
  plain `OpenAI` class used for api.openai.com).

In [1]:
from dotenv import load_dotenv
import os
import sys

from openai import AzureOpenAI

## 2. Load environment variables

`load_dotenv(override=True)` looks for a `.env` file in the current directory and loads its
contents into `os.environ`. `override=True` means values in `.env` take priority over any values
already set in the shell environment.

Make sure you have a `.env` file (in the same folder as this notebook, or on `sys.path`) that
defines:

```
AZURE_APIM_OPENAI_SUBSCRIPTION_KEY=...
AZURE_APIM_OPENAI_API_VERSION=...
AZURE_APIM_OPENAI_ENDPOINT=https://<your-apim-instance>.azure-api.net
AZURE_APIM_OPENAI_DEPLOYMENT=<your-deployment-name>
```

**Note on the endpoint:** it should be the *host only* — e.g.
`https://apim-azr-ue2-bgpt-prd-ucin.azure-api.net` — **without** a trailing
`/openai/deployments/...` path. The SDK builds the full path itself using the API version and
deployment name.

In [2]:
# Read .env and override any existing process env values.
load_dotenv(override=True)

# APIM settings from .env. Endpoint must be the host only, e.g.
# https://apim-azr-ue2-bgpt-prd-ucin.azure-api.net  (no /openai/deployments)
api_key = os.getenv("AZURE_APIM_OPENAI_SUBSCRIPTION_KEY")
api_version = os.getenv("AZURE_APIM_OPENAI_API_VERSION")
endpoint = os.getenv("AZURE_APIM_OPENAI_ENDPOINT")
deployment = os.getenv("AZURE_APIM_OPENAI_DEPLOYMENT")

## 3. Validate configuration

Before making any network calls, confirm that all four required settings were actually found.
`all([...])` returns `False` if any of them is `None` or an empty string, in which case
`sys.exit(...)` prints a helpful message and stops execution (raises `SystemExit` — in a notebook
this will show as an error, which is expected if `.env` is missing or incomplete).

If configuration is present, we print a masked preview of the API key (first 8 characters only)
and the deployment name, just to confirm — without ever logging the key can be verified without
leaking the whole secret.

In [3]:
if not all([api_key, api_version, endpoint, deployment]):
    sys.exit(
        "Missing Azure APIM settings. Set AZURE_APIM_OPENAI_SUBSCRIPTION_KEY, "
        "AZURE_APIM_OPENAI_API_VERSION, AZURE_APIM_OPENAI_ENDPOINT, and "
        "AZURE_APIM_OPENAI_DEPLOYMENT in your .env file."
    )

print(f"Azure APIM key exists and begins {api_key[:8]}")
print(f"Deployment: {deployment}")

Azure APIM key exists and begins 512e1c13
Deployment: gpt-5.1-ptu


## 4. Create the Azure OpenAI client

`AzureOpenAI` is instantiated once and reused for every request. Note the three arguments it
needs, which differ from the plain `OpenAI` client:

- `api_key` — here it's actually the APIM **subscription key**, not an Azure OpenAI resource key,
  since requests are routed through the APIM gateway.
- `api_version` — the Azure OpenAI REST API version (e.g. `2024-06-01`), required because Azure
  versions its API explicitly, unlike api.openai.com.
- `azure_endpoint` — the base host of the APIM instance (see the note above).

In [4]:
# Sync client pointed at Azure APIM.
openai = AzureOpenAI(
    api_key=api_key,
    api_version=api_version,
    azure_endpoint=endpoint,
)

## 5. A small `chat()` helper

This wraps `openai.chat.completions.create(...)` so the rest of the notebook doesn't repeat
boilerplate. Two Azure-specific details are worth calling out:

- **`model=deployment`** — On Azure, the `model` parameter is actually the *deployment name* you
  configured in the Azure OpenAI resource (e.g. `gpt-5-chat`), not a generic model id like
  `gpt-5`. Azure routes the request based on that deployment.
- **`max_completion_tokens` instead of `max_tokens`** — Newer reasoning-capable deployments
  (GPT-5-style) reject the older `max_tokens` parameter and require `max_completion_tokens`
  instead. The helper defaults this to `300` but lets callers override it (used later with `1000` for the longer evaluation step).

In [37]:
# On Azure the `model` argument is the *deployment name*, not an OpenAI model id.
# GPT-5 deployments need max_completion_tokens (max_tokens is rejected).
def chat(messages, deployment=deployment, max_completion_tokens=1000):
    return openai.chat.completions.create(
        model=deployment,
        messages=messages,
        max_completion_tokens=max_completion_tokens,
    )

## 6. Step 1 — Ask for a fun fact

The simplest possible call: a single user message, default token limit (300), and we print the
model's reply text (`response.choices[0].message.content`).

In [38]:
# 1) Fun fact
messages = [{"role": "user", "content": "Tell me a short fun fact"}]
response = chat(messages)
print(response.choices[0].message.content)

Octopuses have three hearts—and their blood is blue.


## 7. Step 2 — Ask the model to invent a hard question

We prompt the model to generate a challenging, IQ-style question, instructing it to respond
**only** with the question itself (no preamble) so that `question` can be reused directly as the
next prompt.

In [39]:
# 2) Ask the model to invent a hard IQ-style question
question = (
    "Please propose a hard, challenging question to assess someone's IQ. "
    "Respond only with the question."
)
messages = [{"role": "user", "content": question}]
response = chat(messages)
question = response.choices[0].message.content
print(question)

A certain 10-digit positive integer uses each digit 0 through 9 exactly once, and if you look at its first \(n\) digits for each \(n=1,2,\dots,10\), that \(n\)-digit number is divisible by \(n\). What is the number?


## 8. Step 3 — Ask the model to answer its own question

The `question` text generated in the previous step is now sent back to the model as a fresh
prompt (a brand-new `messages` list — the model has no memory of generating the question; it's
just answering it as if seeing it cold).

Since we're already in a notebook, we can render the answer as nicely formatted Markdown using
`IPython.display` directly — no fallback needed.

In [40]:
# 3) Ask the model to answer that question
messages = [{"role": "user", "content": question}]
response = chat(messages)
answer = response.choices[0].message.content
print(answer)

The number is:

\[
3816547290
\]

Check the divisibility conditions for each prefix:

- \(3\) is divisible by \(1\)
- \(38\) is divisible by \(2\)
- \(381\) is divisible by \(3\)
- \(3816\) is divisible by \(4\)
- \(38165\) is divisible by \(5\)
- \(381654\) is divisible by \(6\)
- \(3816547\) is divisible by \(7\)
- \(38165472\) is divisible by \(8\)
- \(381654729\) is divisible by \(9\)
- \(3816547290\) is divisible by \(10\)

So the required 10-digit integer is:

\[
\boxed{3816547290}
\]


In [30]:
# Render the answer as Markdown in the notebook.
from IPython.display import Markdown, display

display(Markdown(answer))

Let \(t\) be the number of minutes after 12:00.

- Minute hand angle: \(6t^\circ\)
- Hour hand angle: \(0.5t^\circ\)

We want the angle between them to be \(90^\circ\), so:

\[
|6t - 0.5t| = 90
\]

\[
5.5t = 90
\]

\[
t = \frac{90}{5.5} = \frac{180}{11}
\]

\[
\frac{180}{11} = 16 \frac{4}{11}
\]

So the first time is:

\[
\boxed{16 \frac{4}{11}\text{ minutes after 12:00}}
\]

That is:

\[
\boxed{12:16:\frac{240}{11}\text{ seconds}} = \boxed{12:16:21\frac{9}{11}}
\]

So the first time is **12:16:21 \( \tfrac{9}{11} \)**.

## 9. Step 4 — Ask the model to evaluate the answer

Finally, we build a single prompt string that includes both the `question` and the `answer`
(using an f-string), and ask the model to judge whether the answer is correct. This is a common
"LLM-as-judge" pattern: use the same (or another) model to self-critique its own prior output.

This call uses a higher `max_completion_tokens` (5000 instead of the default 300) since an
evaluation with reasoning tends to need more room than a short fun fact.

In [41]:
# 4) Ask the model to evaluate the answer
deployment="gpt-5.4-ptu"
message = f"""
Here is a question:
{question}

And here is a possible answer that might be correct or incorrect:
{answer}

Please evaluate if the answer is correct or incorrect.
"""
print(message)


Here is a question:
A certain 10-digit positive integer uses each digit 0 through 9 exactly once, and if you look at its first \(n\) digits for each \(n=1,2,\dots,10\), that \(n\)-digit number is divisible by \(n\). What is the number?

And here is a possible answer that might be correct or incorrect:
The number is:

\[
3816547290
\]

Check the divisibility conditions for each prefix:

- \(3\) is divisible by \(1\)
- \(38\) is divisible by \(2\)
- \(381\) is divisible by \(3\)
- \(3816\) is divisible by \(4\)
- \(38165\) is divisible by \(5\)
- \(381654\) is divisible by \(6\)
- \(3816547\) is divisible by \(7\)
- \(38165472\) is divisible by \(8\)
- \(381654729\) is divisible by \(9\)
- \(3816547290\) is divisible by \(10\)

So the required 10-digit integer is:

\[
\boxed{3816547290}
\]

Please evaluate if the answer is correct or incorrect.



In [42]:
messages = [{"role": "user", "content": message}]
response = chat(messages, max_completion_tokens=5000)
print(response.choices[0].message.content)

Correct.

The proposed number is \(3816547290\), which uses each digit \(0\) through \(9\) exactly once.

Check the prefixes:

- \(3\div 1\): yes
- \(38\div 2\): yes
- \(381\div 3\): yes, since \(3+8+1=12\)
- \(3816\div 4\): yes, since last two digits are \(16\)
- \(38165\div 5\): yes, ends in \(5\)
- \(381654\div 6\): yes, divisible by \(2\) and \(3\)
- \(3816547\div 7\): \(3816547 = 7\cdot 545221\)
- \(38165472\div 8\): yes, last three digits \(472\) are divisible by \(8\)
- \(381654729\div 9\): yes, digit sum \(=45\)
- \(3816547290\div 10\): yes, ends in \(0\)

So the answer is correct: \(\boxed{3816547290}\).


## 9. Step 5 — New Challenge




In [44]:
translation_prompt = f"""
You are a professional English-to-Spanish translation agent.

Translate the following answer into clear, natural Spanish.
Keep the orignial meaning, but make it sound fluent for a Spanish-speaking reader.

English text:
{answer}
"""
messages = [{"role": "user", "content": translation_prompt}]
response = chat(messages, deployment="gpt-5.4-ptu", max_completion_tokens=1000)

spanish_translation = response.choices[0].message.content
print(spanish_translation)

El número es:

\[
3816547290
\]

Verificamos las condiciones de divisibilidad para cada prefijo:

- \(3\) es divisible por \(1\)
- \(38\) es divisible por \(2\)
- \(381\) es divisible por \(3\)
- \(3816\) es divisible por \(4\)
- \(38165\) es divisible por \(5\)
- \(381654\) es divisible por \(6\)
- \(3816547\) es divisible por \(7\)
- \(38165472\) es divisible por \(8\)
- \(381654729\) es divisible por \(9\)
- \(3816547290\) es divisible por \(10\)

Por lo tanto, el entero de 10 dígitos requerido es:

\[
\boxed{3816547290}
\]
